## 低频模型构建方法示意图

这个 notebook 面向 PPT 出图，只聚焦单井 `L1-NW1`，生成 3 个并排子图：

- `current_twt`：直接时深转换后的 TWT 域 AI
- `current_twt_lowpass`：低通后的 TWT 域 AI
- 每个地层取 `32` 个点，并按 `lfm_time.py` 的缺失处理逻辑做 `neighbor_slice_fill` 后，再用线性插值连接成趋势曲线

另外会把 `bve_top / bve_bot / itp_bot` 三个层位用虚线同时画在三个子图上。这里**保留了顶底截断**，用来示意当井曲线未完全覆盖层段时，低频模型是如何处理缺失 slice 的。


In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import font_manager

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    raise RuntimeError("Could not locate repo root containing 'src'.")

src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from cup.petrel.load import (
    import_checkshots_petrel,
    import_interpretation_petrel,
    import_well_heads_petrel,
    load_vp_rho_logset_from_las,
)
from cup.seismic.lfm_time import lowpass_twt_log
from cup.seismic.survey import open_survey
from cup.seismic.target_layer import TargetLayer
from wtie.processing import grid

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

WELL_NAME = "L1-NW1"
INTERPRETATION_OFFSET = 0.03
DT_TWT = 0.002
LOWPASS_CUTOFF_HZ = 10.0
LOWPASS_ORDER = 5
N_SLICES_PER_ZONE = 32
WTIE_ROOT = repo_root / "data" / "output_vertical_well_wtie_20260416"


def configure_matplotlib_fonts() -> str | None:
    candidate_fonts = [
        "Microsoft YaHei",
        "SimHei",
        "Noto Sans CJK SC",
        "Source Han Sans SC",
        "PingFang SC",
        "WenQuanYi Zen Hei",
        "Arial Unicode MS",
    ]
    available_fonts = {font.name for font in font_manager.fontManager.ttflist}
    for font_name in candidate_fonts:
        if font_name in available_fonts:
            plt.rcParams["font.family"] = "sans-serif"
            plt.rcParams["font.sans-serif"] = [font_name, *plt.rcParams.get("font.sans-serif", [])]
            plt.rcParams["axes.unicode_minus"] = False
            return font_name
    plt.rcParams["axes.unicode_minus"] = False
    return None


selected_font = configure_matplotlib_fonts()


In [ ]:
def build_target_layer(repo_root: Path, geometry: dict) -> TargetLayer:
    horizon_files = {
        "bve_top": repo_root / "data" / "interpre_time" / "bve_top_t",
        "bve_bot": repo_root / "data" / "interpre_time" / "bve_bot_t",
        "itp_bot": repo_root / "data" / "interpre_time" / "itp_bot_t",
    }

    raw_horizons = {
        horizon_name: import_interpretation_petrel(horizon_file)
        for horizon_name, horizon_file in horizon_files.items()
    }

    return TargetLayer(
        raw_horizon_dfs=raw_horizons,
        geometry=geometry,
        horizon_names=list(horizon_files.keys()),
        outlier_threshold=0.02,
        outlier_min_neighbor_count=2,
    )


def sample_log_with_nan(log: grid.Log, sample_times: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    basis = np.asarray(log.basis, dtype=float)
    values = np.asarray(log.values, dtype=float)
    sample_times = np.asarray(sample_times, dtype=float)
    sampled = np.full(sample_times.shape, np.nan, dtype=float)
    valid_mask = (sample_times >= basis[0]) & (sample_times <= basis[-1])
    sampled[valid_mask] = np.interp(sample_times[valid_mask], basis, values)
    return sampled, valid_mask


def fill_missing_with_neighbors(
    sample_times: np.ndarray, sample_values: np.ndarray
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    filled = np.asarray(sample_values, dtype=float).copy()
    direct_mask = np.isfinite(filled)
    modes = np.where(direct_mask, "direct_sample", "").astype(object)
    valid_indices = np.flatnonzero(direct_mask)
    if valid_indices.size == 0:
        return filled, direct_mask, modes

    for idx in range(filled.size):
        if direct_mask[idx]:
            continue

        prev_candidates = valid_indices[valid_indices < idx]
        next_candidates = valid_indices[valid_indices > idx]
        prev_idx = int(prev_candidates[-1]) if prev_candidates.size else None
        next_idx = int(next_candidates[0]) if next_candidates.size else None

        if prev_idx is None and next_idx is None:
            continue
        if prev_idx is None:
            filled[idx] = filled[next_idx]
            modes[idx] = "neighbor_slice_fill"
            continue
        if next_idx is None:
            filled[idx] = filled[prev_idx]
            modes[idx] = "neighbor_slice_fill"
            continue

        weight = (sample_times[idx] - sample_times[prev_idx]) / (sample_times[next_idx] - sample_times[prev_idx])
        filled[idx] = (1.0 - weight) * filled[prev_idx] + weight * filled[next_idx]
        modes[idx] = "neighbor_slice_fill"

    return filled, direct_mask, modes


def build_layer_piecewise_curve(log: grid.Log, horizon_times: dict[str, float], n_slices: int) -> list[dict]:
    horizon_names = list(horizon_times.keys())
    zone_results = []
    for top_name, bottom_name in zip(horizon_names[:-1], horizon_names[1:]):
        sample_times = np.linspace(horizon_times[top_name], horizon_times[bottom_name], n_slices, dtype=float)
        sample_values, _ = sample_log_with_nan(log, sample_times)
        filled_values, direct_mask, modes = fill_missing_with_neighbors(sample_times, sample_values)
        zone_results.append(
            {
                "top_name": top_name,
                "bottom_name": bottom_name,
                "sample_times": sample_times,
                "sample_values": sample_values,
                "filled_values": filled_values,
                "direct_mask": direct_mask,
                "modes": modes,
            }
        )
    return zone_results


In [ ]:
las_file = repo_root / "data" / "vertical_well_las_target_clear" / f"{WELL_NAME}.las"
tdtable_file = WTIE_ROOT / "tdtable" / f"{WELL_NAME}_{INTERPRETATION_OFFSET}.txt"
seismic_file = repo_root / "data" / "raw" / "mero se 0116_1ms_new_84_coord.Sgy"
well_heads_file = repo_root / "data" / "raw" / "well_heads"

for path in [las_file, tdtable_file, seismic_file, well_heads_file]:
    if not path.exists():
        raise FileNotFoundError(path)

md_ai_log = load_vp_rho_logset_from_las(las_file).AI
td_table_md = import_checkshots_petrel(tdtable_file, depth_domain="md")
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    current_twt = grid.convert_log_from_md_to_twt(md_ai_log, td_table_md, None, DT_TWT)  # type: ignore
current_twt_lowpass = lowpass_twt_log(
    current_twt,
    cutoff_hz=LOWPASS_CUTOFF_HZ,
    order=LOWPASS_ORDER,
)
warning_messages = [str(item.message) for item in caught]

print(f"Selected font: {selected_font}")
print(f"current_twt samples: {len(current_twt)}")
print(f"current_twt_lowpass samples: {len(current_twt_lowpass)}")
if warning_messages:
    print("MD -> TWT warnings:")
    for message in warning_messages:
        print(f"- {message}")


In [ ]:
survey = open_survey(
    seismic_file,
    seismic_type="segy",
    segy_options={"iline": 5, "xline": 21, "istep": 1, "xstep": 4},
)
geometry = survey.query_geometry(domain="time")
target_layer = build_target_layer(repo_root, geometry)

well_heads_df = import_well_heads_petrel(well_heads_file)
head = well_heads_df[well_heads_df["Name"].astype(str).str.strip() == WELL_NAME].iloc[0]
inline, xline = survey.coord_to_line(float(head["Surface X"]), float(head["Surface Y"]))
horizon_times = {
    name: float(value)
    for name, value in target_layer.get_interpretation_values_at_location(float(inline), float(xline)).items()
}
zone_results = build_layer_piecewise_curve(current_twt_lowpass, horizon_times, N_SLICES_PER_ZONE)

zone_summary_df = pd.DataFrame(
    [
        {
            "zone": f"{zone['top_name']}->{zone['bottom_name']}",
            "direct_samples": int(np.count_nonzero(zone["direct_mask"])),
            "neighbor_slice_fill": int(np.count_nonzero((~zone["direct_mask"]) & np.isfinite(zone["filled_values"]))),
        }
        for zone in zone_results
    ]
)

zone_summary_df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 8), sharey=True, constrained_layout=True)

raw_color = "#C44E52"
lowpass_color = "#2A9D8F"
trend_color = "#264653"
direct_color = "#2878B5"
fill_color = "#F4A261"
background_color = "#BFC7D5"
horizon_color = "#4A4A4A"

axes[0].plot(current_twt.values, current_twt.basis, color=raw_color, lw=2.0)
raw_xlim = axes[0].get_xlim()
axes[0].set_title("1. AI log after MD->TWT conversion\n")
axes[0].set_xlabel("AI")
axes[0].set_ylabel("TWT (s)")

axes[1].plot(current_twt_lowpass.values, current_twt_lowpass.basis, color=lowpass_color, lw=2.2)
axes[1].set_title("2. lowpass-filtered log\n(10 Hz cutoff, 5th order Butterworth)")
axes[1].set_xlabel("AI")

axes[2].plot(
    current_twt_lowpass.values,
    current_twt_lowpass.basis,
    color=background_color,
    lw=1.5,
    ls="--",
    alpha=0.9,
    label="current_twt_lowpass",
)
for idx, zone in enumerate(zone_results):
    label_line = "32 points/layer + linear interpolation" if idx == 0 else None
    label_direct = "direct samples" if idx == 0 else None
    label_fill = "neighbor_slice_fill" if idx == 0 else None
    axes[2].plot(zone["filled_values"], zone["sample_times"], color=trend_color, lw=2.4, label=label_line, zorder=3)
    axes[2].scatter(
        zone["filled_values"][zone["direct_mask"]],
        zone["sample_times"][zone["direct_mask"]],
        s=28,
        color=direct_color,
        edgecolor="white",
        linewidth=0.5,
        zorder=4,
        label=label_direct,
    )
    fill_mask = (~zone["direct_mask"]) & np.isfinite(zone["filled_values"])
    if np.any(fill_mask):
        axes[2].scatter(
            zone["filled_values"][fill_mask],
            zone["sample_times"][fill_mask],
            s=42,
            facecolor=fill_color,
            edgecolor="white",
            linewidth=0.7,
            marker="s",
            zorder=5,
            label=label_fill,
        )
axes[2].set_title("3. 32 points per layer\nwith missing-slice fill")
axes[2].set_xlabel("AI")
axes[2].legend(loc="lower right", fontsize=9, frameon=True)

all_horizons = [(name, horizon_times[name]) for name in ["bve_top", "bve_bot", "itp_bot"]]
y_min = min(float(current_twt.basis[0]), min(value for _, value in all_horizons)) - 0.01
y_max = max(float(current_twt.basis[-1]), max(value for _, value in all_horizons)) + 0.01
for ax in axes:
    for _, twt in all_horizons:
        ax.axhline(twt, color=horizon_color, ls=(0, (5, 4)), lw=1.0, alpha=0.8, zorder=1)
    ax.invert_yaxis()
    ax.set_xlim(raw_xlim)
    ax.set_ylim(y_max, y_min)
    ax.grid(True, which="major", axis="both", alpha=0.25)

x_text = axes[2].get_xlim()[1]
for name, twt in all_horizons:
    axes[2].text(x_text, twt, f"  {name}", va="center", ha="left", fontsize=9, color=horizon_color)

fig.suptitle("构建低频模型示意图", fontsize=15, fontweight="bold")

output_dir = repo_root / "data" / "_figures"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"lfm_time_method_diagram_{WELL_NAME}@20260417.png"
fig.savefig(output_path, dpi=220, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {output_path}")
